# C1：FASTSim 整车能耗仿真
# C1: FASTSim Vehicle Energy Simulation

## 一、实验原理 | Principle

### 1.1 什么是整车能耗仿真？
给定一辆车的参数（重量、风阻、电机效率等）和一条速度曲线（工况），
计算跑完这条工况需要多少电。这是电动车续航标定的基础。

### 1.2 FASTSim
**FASTSim**（NREL 美国国家可再生能源实验室开发）：
- 内置 **70+ 真实车型**参数（特斯拉、Leaf 等实测数据）
- **UDDS**：城市拥堵工况（频繁启停）；**HWFET**：高速工况
- 2.x 版本是 Rust 重写的高性能版，本实验调用**真实 FASTSim**

### 1.3 能耗从哪来？
- 加速：动能 `½mv²`
- 爬坡：势能 `mgh`
- 风阻：`½ρCdAv²`（高速时主导）
- 滚阻：`μmg`
- 电机/电池效率损失

## 二、实验目的 | Objectives
1. 掌握 FASTSim 车辆模型库的查询和使用
2. 在 UDDS 工况下仿真纯电动车能耗（kWh/100km）
3. 对比不同车型的能耗，分析影响因素（重量？风阻？）

## 三、学生任务 | Your Tasks
1. **运行**：选一款 BEV，跑 UDDS，记录 kWh/100km
2. **对比**：换 3 款不同重量的车，画柱状图对比
3. **分析**：最重的车能耗最高吗？为什么？（提示：看电机效率 map）
4. **思考题**：为什么高速工况（HWFET）反而比城市工况（UDDS）更省电？（提示：启停 vs 匀速）

## 四、验证内容 | Verification
- [ ] FASTSim 成功导入（注意用 `py310-fastsim` 内核！）
- [ ] 能耗在合理范围（10~20 kWh/100km）
- [ ] 对比了 ≥3 款车型

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `No module named 'fastsim'` | 内核选错了 | 右上角切换到 `Python 3.10 (FASTSim)` 内核 |
| `KeyError: vnum` | 车型编号不存在 | 用 `load_bev_list()` 查可用编号 |
| 能耗为 0 | 仿真未正常运行 | 检查 `SimDrive.sim_drive()` 是否被调用 |


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
from common.fastsim_helper import run_udds, compare_vehicles, plot_cycle, load_bev_list
import matplotlib.pyplot as plt
print('✓ 导入成功')

bevs = load_bev_list()
print(f'车型库：{len(bevs)} 款纯电动车')

In [ ]:
# 画工况
plot_cycle('udds')
plt.show()

In [ ]:
# 跑仿真
vnum = bevs[0][0]
result = run_udds(vnum, 'udds')
energy = result['kwh_per_100km']
print(f'vnum={vnum} 百公里电耗: {energy:.1f} kWh/100km')

# 断言
assert 10.0 <= energy <= 20.0, f'电耗 {energy:.1f} 超出合理范围!'
print('✅ 评测通过！')

In [ ]:
# 对比 3 款
import pandas as pd
df = compare_vehicles([b[0] for b in bevs[:3]], 'udds')
print(df.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.bar(df['vnum'].astype(str), df['kwh_per_100km'])
plt.ylabel('kWh/100km'); plt.title('三款车型能耗对比')
plt.show()

## 🏆 挑战任务
1. 换 HWFET 高速工况跑，对比 UDDS 结果，解释差异。
2. 找一款电耗最低的车型，分析它的参数（车重？风阻？）。